# Chapter 8 — Caches and Message Queues, Part 1
### Memcached examples, runnable cell-by-cell

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/DILDavid/network-programming-notebooks/blob/main/Chapter8_Part1.ipynb)

This notebook reproduces every code example from the lecture slides (slides 6–23)
so you can step through them interactively, modify the parameters, and see the
results visually.

**Sections**
1. Setup — start Memcached, install the client
2. Slide 9 — `set` and `get`
3. Slide 10 — `append`, `incr`, `decr`
4. Slides 11–13 — Cache-aside pattern (squares.py)
5. Slides 14–15 — Keeping cached data fresh (expiry demo)
6. Slides 18–23 — Hashing and sharding comparison

## 1. Setup

You need two things:
1. A running **Memcached server** (default port 11211)
2. The **`python-memcached`** client library

### Start the Memcached server

**Linux**
```bash
sudo apt install -y memcached
sudo systemctl start memcached     # or: memcached -d
```

**Windows**
Install from <https://commaster.net/posts/installing-memcached-windows/>
then run:
```bash
memcached.exe -d start
```

**macOS**
```bash
brew install memcached
brew services start memcached
```

### Install the client

> **Tip.** If `pip` complains about an *externally-managed environment*
> (common on Debian/Ubuntu), create a virtualenv first:
> ```bash
> python3 -m venv .venv && source .venv/bin/activate
> pip install jupyter notebook matplotlib python-memcached
> jupyter notebook
> ```
>
> **Running on Google Colab?** Just run the next cell — it auto-installs
> and starts Memcached for you. No setup needed.

In [ ]:
%pip install --quiet python-memcached

In [ ]:
# Auto-bootstrap: install + start Memcached if it isn't already running.
# Lets this notebook run unmodified on Colab, Binder, fresh VMs, etc.
import shutil
import socket
import subprocess
import time


def memcached_listening(host='127.0.0.1', port=11211, timeout=0.5):
    s = socket.socket()
    s.settimeout(timeout)
    try:
        s.connect((host, port))
        return True
    except (OSError, socket.timeout):
        return False
    finally:
        s.close()


if not memcached_listening():
    if not shutil.which('memcached'):
        print('memcached binary not found — installing via apt …')
        subprocess.run(['sudo', 'apt-get', '-qq', 'update'], check=False)
        r = subprocess.run(['sudo', 'apt-get', '-qq', 'install', '-y', 'memcached'],
                           capture_output=True)
        if r.returncode != 0:
            # Colab and many cloud notebooks run as root; sudo isn't there.
            subprocess.run(['apt-get', '-qq', 'install', '-y', 'memcached'], check=True)
    print('starting memcached on 127.0.0.1:11211 …')
    subprocess.Popen(['memcached', '-u', 'root', '-l', '127.0.0.1', '-p', '11211'])
    time.sleep(1)

assert memcached_listening(), 'Memcached did not come up — see Setup section above.'
print('✓ memcached is reachable on 127.0.0.1:11211')

In [ ]:
import memcache

client = memcache.Client(['127.0.0.1:11211'], debug=0)
stats = client.get_stats()
if stats:
    print("connected to:", stats[0][0].decode() if isinstance(stats[0][0], bytes) else stats[0][0])
    print("memcached version:", stats[0][1].get(b'version', b'?').decode())
else:
    raise RuntimeError(
        "Cannot reach Memcached at 127.0.0.1:11211. "
        "Start it before continuing (see the Setup section above)."
    )

## 2. Slide 9 — `set` and `get`

The simplest possible interaction: store a key, then retrieve it.

In [ ]:
# set and get a Key  -- slide 9
client.set("key01", "value01")
print("key01.value :", client.get("key01"))

> Expected output: `key01.value : value01`
>
> The interface mirrors a Python dictionary, but the storage lives in a
> separate process (Memcached) — possibly on a different machine entirely.

## 3. Slide 10 — `append`, `incr`, `decr`

Memcached supports a few extra primitives beyond plain set/get.

In [ ]:
# append and get a Key  -- slide 10
client.append("key01", ",value02")
print("key01.value :", client.get("key01"))

In [ ]:
client.set("key02", 1)            # new key with value 1
print("key02.value :", client.get("key02"))

In [ ]:
# increment  -- slide 10
client.incr("key02", 100)         # increment by 100
print("key02.value :", client.get("key02"))

In [ ]:
# decrement  -- slide 10
client.decr("key02", 51)          # decrement by 51
print("key02.value :", client.get("key02"))

> Expected output:
> ```
> key01.value : b'value01,value02'
> key02.value : 1
> key02.value : 101
> key02.value : 50
> ```
>
> Note that `incr`/`decr` only work on values that look like integers. If you
> try to `incr` a non-numeric string, Memcached returns `None`.

## 4. Slides 11–13 — Cache-aside pattern (squares.py)

The classic pattern for using a cache:

1. **GET** the result from cache.
2. If **HIT** → return immediately.
3. If **MISS** → compute, **SET** in cache, then return.

We'll use `time.sleep(0.001)` to emulate an "expensive" operation, then run
many iterations and watch the cumulative time per round drop as the cache
warms up.

In [ ]:
import random
import time
import timeit


def compute_square(mc, n):
    value = mc.get('sq:%d' % n)
    if value is None:                    # cache miss
        time.sleep(0.001)                # emulate expensive operation
        value = n * n
        mc.set('sq:%d' % n, value)
    return value


def make_request():
    compute_square(client, random.randint(0, 5000))


# Start cold every time you re-run this cell, so you see the warm-up effect.
client.flush_all()

print('Ten successive runs:')
times = []
for i in range(1, 11):
    t = timeit.timeit(make_request, number=2000)
    times.append(t)
    print(f' {t:.2f}s', end='', flush=True)
print()

**First time you run this cell**, expect output like:
```
25.59s 17.03s 11.76s 7.88s 4.73s 3.27s 2.46s 1.58s 1.09s 0.88s
```

**Re-run the cell** and you should see something much faster:
```
0.47s 0.42s 0.31s 0.28s 0.26s 0.24s 0.23s 0.21s 0.22s 0.20s
```

Why? The first run mostly missed the cache, so `time.sleep(0.001)` fired for
nearly every request. By the second run, the cache holds answers for most of
the random integers in `[0, 5000]`.

Let's visualise it.

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(10, 4))
bars = ax.bar(range(1, len(times) + 1), times)
bars[0].set_color('#f5b955')             # cold cache — amber
for b in bars[1:]:
    b.set_color('#4dd6e6')               # warming — cyan
ax.set_xlabel('round')
ax.set_ylabel('seconds (lower is better)')
ax.set_title('Cache-aside speedup over 10 rounds')
ax.set_xticks(range(1, 11))
plt.tight_layout()
plt.show()

if times:
    print(f"first round: {times[0]:.2f}s   last round: {times[-1]:.2f}s   "
          f"speedup: {times[0]/times[-1]:.1f}x")

## 5. Slides 14–15 — Keeping cached data fresh

Three strategies:

1. **Expiration** — `client.set(key, value, time=N)` makes Memcached drop the
   entry after `N` seconds.
2. **Active invalidation** — call `client.delete(key)` when the underlying
   data changes.
3. **Replacement** — overwrite a popular invalidated entry with a fresh
   `client.set` so concurrent readers don't all stampede to recompute it.

Below we demonstrate strategy 1 (expiry) live. Strategies 2 and 3 are just
`delete`/`set` calls and need no special demo.

In [ ]:
import time

client.set("ephemeral", "I will vanish", time=2)   # 2-second TTL
print("now           :", client.get("ephemeral"))
time.sleep(2.5)
print("after 2.5s    :", client.get("ephemeral"))   # -> None

## 6. Slides 18–23 — Hashing and sharding

When you have **multiple Memcached instances**, the client picks which server
holds each key by computing a number from the key. We compare three ways of
doing that.

In [ ]:
import hashlib


def alpha_shard(word):
    """Slide 20 — manual alphabetic ranges."""
    if word[0] < 'g':       # abcdef
        return 'server0'
    elif word[0] < 'n':     # ghijklm
        return 'server1'
    elif word[0] < 't':     # nopqrs
        return 'server2'
    else:                   # tuvwxyz
        return 'server3'


def hash_shard(word):
    """Slide 21 — Python's built-in hash()."""
    return 'server%d' % (hash(word) % 4)


def md5_shard(word):
    """Slide 21 — MD5 cryptographic hash."""
    data = word.encode('utf-8')
    return 'server%d' % (hashlib.md5(data).digest()[-1] % 4)

Now download the lecture's word list (~104k English words). The cell below
fetches it on first run; afterwards it's cached as `words.txt` next to the
notebook.

In [ ]:
import os
import urllib.request

WORDS_URL = 'https://raw.githubusercontent.com/dwyl/english-words/master/words.txt'

if not os.path.exists('words.txt'):
    print('downloading words.txt …')
    urllib.request.urlretrieve(WORDS_URL, 'words.txt')

words = open('words.txt').read().split()
print(f"{len(words):,} words loaded")

In [ ]:
# Slide 22 — distribute every word across 4 servers using each algorithm.
results = {}
for fn in (alpha_shard, hash_shard, md5_shard):
    counts = {f'server{i}': 0 for i in range(4)}
    for w in words:
        wl = w.lower()
        if wl and wl[0].isalpha():
            counts[fn(wl)] += 1
    name = fn.__name__.replace('_shard', '')
    results[name] = counts
    print(name)
    for key, value in sorted(counts.items()):
        print(f'  {key} {value} {value/len(words):.2f}')
    print()

**Expected pattern (slide 23):**

```
alpha     # uneven  -- the data's structure leaks through
  server0 ~150k  0.32
  server1 ~98k   0.21
  server2 ~147k  0.32
  server3 ~72k   0.15

hash      # even  -- 25% / 25% / 25% / 25%
md5       # even  -- 25% / 25% / 25% / 25%
```

Visually:

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 4), sharey=True)
for ax, (name, counts) in zip(axes, results.items()):
    keys = sorted(counts.keys())
    values = [counts[k] for k in keys]
    color = '#f5b955' if name == 'alpha' else '#4dd6e6'
    ax.bar(keys, values, color=color, edgecolor=color, alpha=0.8)
    ax.set_title(name, fontsize=14, fontweight='bold')
    ax.set_xlabel('shard')

    # imbalance ratio (max / min, perfect = 1.0)
    imb = max(values) / max(min(values), 1)
    ax.text(0.5, 0.92, f'imbalance {imb:.2f}',
            ha='center', transform=ax.transAxes,
            fontfamily='monospace',
            color='#444')

axes[0].set_ylabel('words assigned')
fig.suptitle('Sharding: distributing 100k+ words across 4 servers', y=1.02)
plt.tight_layout()
plt.show()

## Takeaway

- **Manual ranges** (Algorithm 1) leak the structure of your data — letter
  frequencies in this case — and produce hot shards.
- **Hashing** (Algorithms 2 and 3) treats keys uniformly and distributes load
  evenly across servers, so you don't get hotspots.
- Sharding should always be done with hashing, unless your storage system
  handles partitioning for you.

---

### Next steps

- Try changing the number of shards in `alpha_shard` / `hash_shard` /
  `md5_shard` from 4 to 8 or 16 — the imbalance gap widens.
- Replace `time.sleep(0.001)` in the squares demo with a heavier operation
  (e.g. a database lookup) and observe how the speedup compounds.
- Continue to Part 2 of the chapter: **Message Queues**.